# **TASK 04 — GRAPH NEURAL NETWORK DEVELOPMENT**

## **Imports & Environment Setup**

In [1]:
import os
import sys
import time
import pickle
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.3.1+cpu
CUDA available: False


## **Reproducibility & Device Setup**

In [2]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Fix all relevant random seeds for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


## **Load Processed Data**

In [3]:
def find_project_root(marker: str = "requirements.txt") -> Path:
    """Walk up from the current working directory until we find
    the project root, identified by the presence of a known file."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not locate project root (missing {marker})")

PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT / "src"))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed_data.pkl"

with open(DATA_PATH, "rb") as f:
    processed = pickle.load(f)

features   = processed["features"]      # shape: [num_nodes, 128]
labels     = processed["labels"]        # shape: [num_nodes]
edge_index = processed["edge_index"]    # shape: [2, num_edges]
train_idx  = processed["train_idx"]
valid_idx  = processed["valid_idx"]
test_idx   = processed["test_idx"]

num_classes = int(labels.max().item()) + 1
num_features = features.shape[1]

# Wrap everything in a single PyG Data object — the standard
# container GNN layers expect (x, edge_index, y together).
graph_data = Data(
    x=features,
    edge_index=edge_index,
    y=labels,
).to(DEVICE)

print(f"Nodes: {graph_data.num_nodes:,}")
print(f"Edges: {graph_data.num_edges:,}")
print(f"Features per node: {num_features}")
print(f"Number of classes: {num_classes}")
print(f"Train / Valid / Test sizes: {len(train_idx):,} / {len(valid_idx):,} / {len(test_idx):,}")

Nodes: 169,343
Edges: 1,166,243
Features per node: 128
Number of classes: 40
Train / Valid / Test sizes: 90,941 / 29,799 / 48,603


## **Model Architectures**

GCN and GraphSAGE (the two required architectures) and GAT (a third,
bonus architecture) are defined in `src/models/gnn_models.py` — a
shared, importable module, not redefined locally in this notebook.
This ensures every downstream task (training in Task 05, explainability
in Task 07) always uses the exact same architecture as demonstrated
below, with no risk of manual copy-paste drift between notebooks.

In [4]:
from models.gnn_models import GCN, GraphSAGE, GAT

print("Imported GCN, GraphSAGE, and GAT from src/models/gnn_models.py")
print("\nGCN docstring:")
print(GCN.__doc__)
print("\nGraphSAGE docstring:")
print(GraphSAGE.__doc__)
print("\nGAT docstring:")
print(GAT.__doc__)

Imported GCN, GraphSAGE, and GAT from src/models/gnn_models.py

GCN docstring:

    Baseline spectral-style GNN (Kipf & Welling, 2017).

    Architecture:
        Input -> GCNConv -> BatchNorm -> ReLU -> Dropout
              -> GCNConv -> BatchNorm -> ReLU -> Dropout
              -> GCNConv -> logits

    Design choices:
        - 3 layers: enough to aggregate 3-hop neighborhood info
          without over-smoothing (a known GCN failure mode at
          higher depth on citation graphs).
        - BatchNorm after each conv, before activation: stabilizes
          training and matches the standard OGB baseline
          implementation for this dataset.
        - ReLU: standard, cheap, avoids vanishing gradients.
        - Dropout after each hidden layer: regularizes against
          overfitting on the small labeled fraction of nodes.
    

GraphSAGE docstring:

    Inductive, neighbor-sampling-friendly GNN (Hamilton et al., 2017).

    Architecture:
        Input -> SAGEConv -> Batch

### **Model 3: Graph Attention Network (GAT) — Bonus: Advanced GNN Architecture**

A third architecture, added beyond the two required models (GCN and
GraphSAGE), to attempt the **"Advanced GNN Architectures"** bonus
category. GAT also enables genuine attention-weight-based
explainability in Task 07 (Option C: Attention Weight Analysis),
which GCN and GraphSAGE cannot provide since neither has a learned
per-neighbor importance score.

See `GAT.forward_with_attention()` in `src/models/gnn_models.py` for
the method that exposes raw attention weights for downstream analysis.

### **Hyperparameter Configuration & Model Instantiation**

In [5]:
CONFIG = {
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": num_features,
    "out_channels": num_classes,
}

gcn_model = GCN(**CONFIG).to(DEVICE)
sage_model = GraphSAGE(**CONFIG).to(DEVICE)
gat_model = GAT(**CONFIG, heads=4).to(DEVICE)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("GCN trainable parameters:      ", f"{count_parameters(gcn_model):,}")
print("GraphSAGE trainable parameters:", f"{count_parameters(sage_model):,}")
print("GAT trainable parameters:      ", f"{count_parameters(gat_model):,}")

GCN trainable parameters:       110,120
GraphSAGE trainable parameters: 218,664
GAT trainable parameters:       177,272


### **Sanity Check: Forward Pass Verification**

In [6]:
gcn_model.eval()
sage_model.eval()
gat_model.eval()

with torch.no_grad():
    start = time.time()
    gcn_out = gcn_model(graph_data.x, graph_data.edge_index)
    gcn_time = time.time() - start

    start = time.time()
    sage_out = sage_model(graph_data.x, graph_data.edge_index)
    sage_time = time.time() - start

    start = time.time()
    gat_out = gat_model(graph_data.x, graph_data.edge_index)
    gat_time = time.time() - start

for name, out in [("GCN", gcn_out), ("GraphSAGE", sage_out), ("GAT", gat_out)]:
    assert out.shape == (graph_data.num_nodes, num_classes), f"{name} output shape mismatch"

print(f"GCN output shape:       {tuple(gcn_out.shape)}  |  forward pass: {gcn_time:.2f}s")
print(f"GraphSAGE output shape: {tuple(sage_out.shape)}  |  forward pass: {sage_time:.2f}s")
print(f"GAT output shape:       {tuple(gat_out.shape)}  |  forward pass: {gat_time:.2f}s")
print("\nAll three models produce correctly-shaped logits — ready for training in Task 05.")

GCN output shape:       (169343, 40)  |  forward pass: 3.38s
GraphSAGE output shape: (169343, 40)  |  forward pass: 2.19s
GAT output shape:       (169343, 40)  |  forward pass: 27.43s

All three models produce correctly-shaped logits — ready for training in Task 05.


### **Save Model Configs for Task 05 Handoff**

In [7]:
import json

CONFIG_PATH = PROJECT_ROOT / "models_checkpoints" / "model_config.json"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

model_config = {
    "gcn": {"class": "GCN", **CONFIG},
    "graphsage": {"class": "GraphSAGE", **CONFIG},
    "gat": {"class": "GAT", **CONFIG, "heads": 4},
    "seed": SEED,
    "model_module": "src/models/gnn_models.py",
    "bonus_category": "Advanced GNN Architectures (GAT added as 3rd model)",
    "usage_note": "Import architectures with: from models.gnn_models import GCN, GraphSAGE, GAT. Then: model = GCN(**config['gcn'])"
}

with open(CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=2)

print(f"Saved model_config.json to: {CONFIG_PATH}")
print(json.dumps(model_config, indent=2))

Saved model_config.json to: C:\Users\thrit\Desktop\ogbn-arxiv-gnn\models_checkpoints\model_config.json
{
  "gcn": {
    "class": "GCN",
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": 128,
    "out_channels": 40
  },
  "graphsage": {
    "class": "GraphSAGE",
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": 128,
    "out_channels": 40
  },
  "gat": {
    "class": "GAT",
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": 128,
    "out_channels": 40,
    "heads": 4
  },
  "seed": 42,
  "model_module": "src/models/gnn_models.py",
  "bonus_category": "Advanced GNN Architectures (GAT added as 3rd model)",
  "usage_note": "Import architectures with: from models.gnn_models import GCN, GraphSAGE, GAT. Then: model = GCN(**config['gcn'])"
}
